# 4C-SCF QOL improvements
In the previous notebook, we were able to run a few iterations of a 4c-scf in He. However there are some things that must be adressed to be directly extracted, such as the size of the basis, eri tensor cleanup and occupation definition. 

Also we will start moving towards a context class similar to the previously established one in the NR-(CS)SCF code. 

In [1]:
from typing import Union
from dataclasses import dataclass

import numpy as np
from numpy.typing import NDArray

from py_mods.src.external.DIRAC_ME import (
    build_S_V_W_T_from_h5,
    get_nuc_charge,
    full_eri_from_h5,
    build_uncontracted_basis_from_h5,
)

from py_mods.src.SCF_4c_dev.KUSCF_dev import (
    occupation_4c,
    eri_classified,
)

Taking as a reference the `CSRHFContext` class:

In [2]:
@dataclass
class _primitive_KUSCFContext:
    n_bas: int
    nL: int
    nS: int
    S: NDArray[np.float64]
    T: NDArray[np.complex128]
    V: NDArray[np.float64]
    W: NDArray[np.float64]
    eri_classess: NDArray[np.float64]
    n_electrons: int

    # Optional
    occupation: Union[int, NDArray[np.uint8], None] = None

In [3]:
def generate_primitive_KUSCFContext_from_h5(
    h5_filename: str,
) -> _primitive_KUSCFContext:

    S, V, W, T = build_S_V_W_T_from_h5(h5_filename)
    H_nuc = V + W + T
    nuc_charge = get_nuc_charge(h5_filename)

    _, nL, nS = build_uncontracted_basis_from_h5(h5_filename)
    eri = full_eri_from_h5(h5_filename)
    eri = eri_classified(eri, nL)

    occ_det = occupation_4c(nS, nL, 2)

    return _primitive_KUSCFContext(
        n_bas=nL * nS,
        nL=nL,
        nS=nS,
        S=S,
        T=T,
        V=V,
        W=W,
        eri_classess=eri,
        n_electrons=nuc_charge,
        occupation=occ_det,
    )

In [4]:
He_context = generate_primitive_KUSCFContext_from_h5("data/He_checkpoint.h5")